# STEP 1: Overview of Script
# ---------------------------------------
# This script automates the assignment of window constructions in ESP-r models.
#
# Workflow:
# 1. Load EPC dataset (contains BUILDING_REFERENCE_NUMBER and ROOF_CONS).
# 2. Load mapping file (WINDOW_CONS → ESP-r assignment characters).
#    - Handles both single-letter and multi-line assignments (e.g. "0\nz").
# 3. Loop through each building folder in baseline_models.
# 4. Identify the model subfolder (e.g. Detached_2F, Flat, etc.).
# 5. Locate the cfg/windows.txt file.
# 6. Replace all instances of "ROOF" in roof.txt with the correct assignment.
# 7. Save the updated file.
#
# Notes:
# - Buildings without roof.txt are skipped.
# - Multi-line assignments preserve correct line breaks (critical for ESP-r).
# - Designed for use in Jupyter Notebook on macOS.

In [ ]:
# STEP 2: Import Libraries and Define Paths
# ---------------------------------------
import os
import pandas as pd

# File paths
epc_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_NPV_IRR_update.csv"
mapping_path = "/Users/jakegehrung/Desktop/data_raw/REFERENCE FILES/window_ESP-r_assign.csv"
baseline_dir = "/Users/jakegehrung/Desktop/data_raw/baseline_models"

In [2]:
# STEP 3: Load EPC Dataset
# ---------------------------------------
epc_df = pd.read_csv(epc_path)

# Ensure correct columns exist
required_cols = ["BUILDING_REFERENCE_NUMBER", "WINDOW_CONS"]
for col in required_cols:
    if col not in epc_df.columns:
        raise ValueError(f"Missing column: {col}")

# Convert building ID to string for matching folder names
epc_df["BUILDING_REFERENCE_NUMBER"] = epc_df["BUILDING_REFERENCE_NUMBER"].astype(str)

print(f"EPC dataset loaded: {len(epc_df)} records")

EPC dataset loaded: 117 records


In [3]:
# STEP 4 (FINAL FIX): Proper CSV Parsing with Multi-line Support
# ---------------------------------------
mapping_dict = {}

with open(mapping_path, "r") as f:
    content = f.read()

# Split into logical CSV rows (handle quoted multiline properly)
rows = []
current = ""
in_quotes = False

for char in content:
    if char == '"':
        in_quotes = not in_quotes

    if char == "\n" and not in_quotes:
        rows.append(current.strip())
        current = ""
    else:
        current += char

# Add last row
if current:
    rows.append(current.strip())

# Remove header
rows = rows[1:]

# Parse rows
for row in rows:
    if "," not in row:
        continue

    key, value = row.split(",", 1)

    key = key.strip()
    value = value.strip()

    # Remove surrounding quotes if present
    if value.startswith('"') and value.endswith('"'):
        value = value[1:-1]

    # Replace literal newlines correctly
    value = value.replace("\r", "").strip()

    mapping_dict[key] = value

# Debug check
print(f"Loaded {len(mapping_dict)} mappings:\n")
for k, v in list(mapping_dict.items())[:10]:
    print(f"{k} -> {repr(v)}")

Loaded 4 mappings:

dbl_glz_pre_2003 -> 'm'
triple_glz -> 'o'
dbl_glz_post_2003 -> 'n'
single_glz -> 'l'


In [4]:
#VALIDATION CHECK: Ensure all keys in mapping_dict exist in EPC dataset
print("\nTEST LOOKUPS:")
test_keys = [
    "dbl_glz_pre_2003",
    "triple_glz"
]

for key in test_keys:
    print(f"{key} -> {repr(mapping_dict.get(key))}")


TEST LOOKUPS:
dbl_glz_pre_2003 -> 'm'
triple_glz -> 'o'


In [ ]:
# STEP 5: Define Possible Model Folder Names
# ---------------------------------------
model_folder_names = [
    "SemiDetached_2F", "Detached_2F"
]

In [6]:
# STEP 6: Process Each Building
# ---------------------------------------
processed = 0
skipped = 0

for _, row in epc_df.iterrows():
    building_id = row["BUILDING_REFERENCE_NUMBER"]
    window_cons = row["WINDOW_CONS"]

    # Get assignment from mapping
    assignment = mapping_dict.get(window_cons)

    if assignment is None:
        print(f"Skipping {building_id}: No mapping for {window_cons}")
        skipped += 1
        continue

    building_path = os.path.join(baseline_dir, building_id)

    if not os.path.isdir(building_path):
        print(f"Skipping {building_id}: Folder not found")
        skipped += 1
        continue

    # Find model folder
    model_folder = None
    for name in model_folder_names:
        candidate = os.path.join(building_path, name)
        if os.path.isdir(candidate):
            model_folder = candidate
            break

    if model_folder is None:
        print(f"Skipping {building_id}: No model folder found")
        skipped += 1
        continue

    # Path to windows.txt
    windows_txt_path = os.path.join(model_folder, "cfg", "windows.txt")

    if not os.path.isfile(windows_txt_path):
        print(f"Skipping {building_id}: windows.txt not found")
        skipped += 1
        continue

    # Read file
    with open(windows_txt_path, "r") as f:
        content = f.read()

    # Replace placeholder
    updated_content = content.replace("WINDOW", assignment)

    # Save file
    with open(windows_txt_path, "w") as f:
        f.write(updated_content)

    processed += 1

print("\n--- Processing Complete ---")
print(f"Processed: {processed}")
print(f"Skipped: {skipped}")

Skipping 1001829067: windows.txt not found
Skipping 1001951858: windows.txt not found
Skipping 1001829071: windows.txt not found
Skipping 1234761001: windows.txt not found
Skipping 1001991633: windows.txt not found
Skipping 1001829059: windows.txt not found
Skipping 1001063639: windows.txt not found
Skipping 1234761000: windows.txt not found
Skipping 1236594950: windows.txt not found
Skipping 1001664925: windows.txt not found
Skipping 1001906271: windows.txt not found
Skipping 1000238911: windows.txt not found
Skipping 1001829057: windows.txt not found
Skipping 1234760999: windows.txt not found
Skipping 1002143357: windows.txt not found
Skipping 1001951854: windows.txt not found
Skipping 1001829069: windows.txt not found
Skipping 1002313096: windows.txt not found
Skipping 1002143351: windows.txt not found
Skipping 1001870854: windows.txt not found
Skipping 1001870864: windows.txt not found
Skipping 1002143293: windows.txt not found
Skipping 1002143352: windows.txt not found
Skipping 10

In [7]:
# STEP 7: Optional Validation (Check a Few Files)
# ---------------------------------------
# Randomly print a few updated files to verify correctness
import random

sample_ids = random.sample(list(epc_df["BUILDING_REFERENCE_NUMBER"]), 5)

for building_id in sample_ids:
    building_path = os.path.join(baseline_dir, building_id)

    for name in model_folder_names:
        model_folder = os.path.join(building_path, name)
        windows_txt_path = os.path.join(model_folder, "cfg", "windows.txt")

        if os.path.isfile(windows_txt_path):
            print(f"\n--- {building_id} ---")
            with open(windows_txt_path, "r") as f:
                print(f.read())
            break